<a href="https://colab.research.google.com/github/mchadw/projSwing/blob/main/projSwing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
matches_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-matches.csv')
points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')
points_2010s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv')

points_raw = pd.concat([points_2010s_raw, points_2020s_raw])

matches_raw.head(10)
matches_raw.shape

/tmp/ipykernel_2908/1195214542.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')


(7566, 15)

In [ ]:
points_raw.head(10)

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
0,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,1,0,0,0,0.0,0-0,1,True,1,4n,4b1b1b3n@,NaN,1
1,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,2,0,0,0,0.0,15-0,1,True,1,5n,5f28f2b1b1b2f3f3d@,NaN,1
2,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,3,0,0,0,0.0,30-0,1,True,1,4C,NaN,NaN,1
3,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,4,0,0,0,0.0,40-0,1,True,1,4r2d#,NaN,NaN,1
4,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,5,0,0,1,0.0,0-0,2,True,2,5d,c4b1b1d!@,NaN,1
5,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,6,0,0,1,0.0,0-15,2,True,2,4n,4f38f2b1b1b1b1w@,NaN,1
6,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,7,0,0,1,0.0,0-30,2,True,2,6f#,NaN,NaN,2
7,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,8,0,0,1,0.0,15-30,2,True,2,4f38f+3f1i^1i3n#,NaN,NaN,2
8,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,9,0,0,1,0.0,30-30,2,True,2,4w,6f17f3f3f3n#,NaN,1
9,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,10,0,0,1,0.0,30-40,2,True,2,4r17b1m3o3*,NaN,NaN,2


In [ ]:
import random
import re

PLAYER = "Carlos Alcaraz"

MATCHES_PATH = "/content/drive/MyDrive/projSwing/charting-m-matches.csv"
POINTS_2020S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv"
POINTS_2010S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv"

matches_raw = pd.read_csv(MATCHES_PATH)
points_2020s_raw = pd.read_csv(POINTS_2020S_PATH, low_memory=False)
points_2010s_raw = pd.read_csv(POINTS_2010S_PATH, low_memory=False)
points_raw = pd.concat([points_2010s_raw, points_2020s_raw], ignore_index=True)

p1_is_player = matches_raw["Player 1"].fillna("").str.contains(PLAYER, case=False, regex=False)
p2_is_player = matches_raw["Player 2"].fillna("").str.contains(PLAYER, case=False, regex=False)
# Keep matches where PLAYER is on exactly one side.
keep = p1_is_player ^ p2_is_player
player_matches = matches_raw.loc[keep, ["match_id", "Player 1", "Player 2"]].copy()
player_matches["player_side"] = 2
player_matches.loc[p1_is_player.loc[keep], "player_side"] = 1

match_meta = player_matches[["match_id", "player_side"]].drop_duplicates("match_id")
player_points = points_raw.merge(match_meta, on="match_id", how="inner")

print(f"{PLAYER}: {match_meta['match_id'].nunique()} matches, {len(player_points)} points")

FH = set("frvhjulo")
BH = set("bszikymp")
SHOT = re.compile(r"([fbrsvzopuylmhijktq])[+\-=;^]*([0-3])?[789]?[nwdxe!g]*[*#@C]?")
SERVE_ONLY = re.compile(r"^c*[0456]?\+?[nwdxegV!]+$")
SERVE_PREFIX = re.compile(r"^c*[0456]?\+?")
LABELS = ["f1", "f2", "f3", "b1", "b2", "b3"]


def rally(first, second):
    """Return (shot buckets, rally_starts_with_return).

    A bucket is f/b plus direction 1-3, or None when the token is not one of those
    six categories. The serve number is not a bucket. The first bucket is the
    returner's shot. Depth codes 7-9 are intentionally omitted from this basic model.
    """
    code = first if isinstance(first, str) else ""
    if SERVE_ONLY.fullmatch(code):
        code = second if isinstance(second, str) else ""
    code = SERVE_PREFIX.sub("", code)
    starts_with_return = True
    shots = []
    for stroke, direction in SHOT.findall(code):
        hand = "f" if stroke in FH else "b" if stroke in BH else None
        shots.append(
            hand + direction
            if hand and direction in ("1", "2", "3")
            else None
        )
    return shots, starts_with_return


def hitters_for(n_shots, server, starts_with_return):
    """Player index (1 or 2) for each parsed shot, alternating from the first hitter."""
    server = int(server)
    returner = 3 - server
    hitter = returner if starts_with_return else server
    out = []
    for _ in range(n_shots):
        out.append(hitter)
        hitter = 3 - hitter
    return out


def response_detail(first, second, svr, player_side):
    """Return parsed shots and all opponent-to-player response opportunities.

    Candidate pairs preserve missing shot buckets. Eligible pairs are the subset for
    which both the incoming shot and the player's response belong to LABELS. Keeping
    these populations separate prevents model coverage from being confused with
    predictive accuracy.
    """
    shots, starts_with_return = rally(first, second)
    try:
        server = int(svr)
        side = int(player_side)
    except (TypeError, ValueError):
        return shots, [], [], []
    if server not in (1, 2) or side not in (1, 2):
        return shots, [], [], []

    who = hitters_for(len(shots), server, starts_with_return)
    opponent_side = 3 - side
    candidate_pairs = []
    for i in range(len(shots) - 1):
        # Eligibility is determined by the sequence of hitters, not by whether the
        # shot labels happen to be observable. This retains an auditable denominator.
        if who[i] == opponent_side and who[i + 1] == side:
            candidate_pairs.append((shots[i], shots[i + 1]))

    eligible_pairs = [
        (incoming, reply)
        for incoming, reply in candidate_pairs
        if incoming is not None and reply is not None
    ]
    return shots, who, candidate_pairs, eligible_pairs


def extract_response_data(points, n_examples=8, seed=0):
    """Create the analysis population and document losses caused by missing labels.

    The returned data frame has one row per eligible transition. It is the single
    source from which training and test tables should be drawn, ensuring that all
    models are compared on the same empirical observations.
    """
    rng = random.Random(seed)
    stats = {
        "points": 0,
        "empty_points": 0,
        "points_with_candidate": 0,
        "points_with_eligible": 0,
        "candidate_pairs": 0,
        "eligible_pairs": 0,
        "missing_incoming": 0,
        "missing_reply": 0,
        "missing_both": 0,
    }
    records = []
    examples = []

    columns = ["match_id", "Pt", "1st", "2nd", "Svr", "player_side"]
    for match_id, point_number, first, second, svr, side in points[columns].itertuples(
        index=False, name=None
    ):
        stats["points"] += 1
        shots, who, candidates, eligible = response_detail(first, second, svr, side)

        if not shots:
            stats["empty_points"] += 1
        if candidates:
            stats["points_with_candidate"] += 1
        if eligible:
            stats["points_with_eligible"] += 1

        for incoming, reply in candidates:
            stats["candidate_pairs"] += 1
            if incoming is None and reply is None:
                stats["missing_both"] += 1
            elif incoming is None:
                stats["missing_incoming"] += 1
            elif reply is None:
                stats["missing_reply"] += 1
            else:
                stats["eligible_pairs"] += 1

        for transition_number, (incoming, reply) in enumerate(eligible, start=1):
            records.append(
                {
                    "match_id": match_id,
                    "point_number": point_number,
                    "transition_number": transition_number,
                    "incoming": incoming,
                    "actual_response": reply,
                }
            )

        if eligible:
            example = {
                "match_id": match_id,
                "point_number": point_number,
                "1st": first,
                "2nd": second,
                "Svr": svr,
                "player_side": side,
                "shots": shots,
                "hitters": who,
                "candidates": [f"{a}->{b}" for a, b in candidates],
                "eligible": [f"{a}->{b}" for a, b in eligible],
            }
            # Reservoir sampling avoids privileging observations that occur early in
            # the source files while retaining a small, reproducible audit sample.
            if len(examples) < n_examples:
                examples.append(example)
            else:
                j = rng.randint(0, stats["points_with_eligible"] - 1)
                if j < n_examples:
                    examples[j] = example

    response_data = pd.DataFrame.from_records(
        records,
        columns=[
            "match_id",
            "point_number",
            "transition_number",
            "incoming",
            "actual_response",
        ],
    )
    return response_data, stats, examples


def build_response_matrix(response_data):
    """Count eligible incoming-shot -> player-response transitions."""
    matrix = pd.crosstab(
        response_data["incoming"], response_data["actual_response"], dropna=False
    ).reindex(index=LABELS, columns=LABELS, fill_value=0)
    matrix.index.name = "opponent_shot"
    matrix.columns.name = "player_response"
    return matrix


def make_evaluation_table(test_pairs, matrix_guesses, baseline_guesses):
    """Place both models' predictions beside the same held-out outcomes.

    Predictions are attached positionally only after verifying equal lengths. No
    observations are removed here: model-specific row filtering would make accuracy
    estimates incomparable. The policy for unavailable predictions is addressed
    separately when the models are fitted.
    """
    n_rows = len(test_pairs)
    if len(matrix_guesses) != n_rows or len(baseline_guesses) != n_rows:
        raise ValueError("Each model must provide one prediction per test transition")

    evaluation = test_pairs.reset_index(drop=True).copy()
    evaluation["matrix_guess"] = list(matrix_guesses)
    evaluation["baseline_guess"] = list(baseline_guesses)
    return evaluation


def _check_parser():
    shots, who, candidates, eligible = response_detail("4n", "4b1f2n@", 1, 1)
    assert shots == ["b1", "f2"], shots
    assert who == [2, 1], who
    assert candidates == [("b1", "f2")], candidates
    assert eligible == [("b1", "f2")], eligible

    shots, who, candidates, eligible = response_detail("4n", "4f1b2f3", 1, 2)
    assert shots == ["f1", "b2", "f3"], shots
    assert who == [2, 1, 2], who
    assert candidates == [("b2", "f3")], candidates
    assert eligible == [("b2", "f3")], eligible

    shots, who, candidates, eligible = response_detail("4C", None, 1, 1)
    assert shots == [] and candidates == [] and eligible == []

    shots, who, candidates, eligible = response_detail("6f1b2*", None, 1, 1)
    assert shots == ["f1", "b2"], shots
    assert who == [2, 1], who
    assert candidates == [("f1", "b2")], candidates
    assert eligible == [("f1", "b2")], eligible

    # A transition with an unclassified incoming shot remains in the candidate
    # population but is excluded from the six-category analysis population. This
    # distinction is required to calculate and report empirical coverage.
    shots, who, candidates, eligible = response_detail("6f1f0b2", None, 2, 1)
    assert shots[0] == "f1" and shots[1] is None and shots[2] == "b2"
    assert candidates == [(None, "b2")], candidates
    assert eligible == [], eligible


_check_parser()
print("parser, hitter, and eligibility checks passed")

response_data, stats, examples = extract_response_data(player_points)
matrix = build_response_matrix(response_data)

# Coverage is reported at the transition level because transition-level accuracy is
# the primary outcome. Point-level summaries remain useful descriptive diagnostics,
# but they do not share the denominator used to estimate predictive performance.
candidate_n = stats["candidate_pairs"]
eligible_n = stats["eligible_pairs"]
transition_coverage = eligible_n / candidate_n if candidate_n else float("nan")
point_n = stats["points"] or 1

print(f"matches represented in point data: {player_points['match_id'].nunique()}")
print(f"points: {stats['points']}")
print(f"candidate opponent -> {PLAYER} transitions: {candidate_n}")
print(f"eligible six-category transitions: {eligible_n}")
print(f"transition coverage: {transition_coverage:.1%}")
print("exclusions from the six-category analysis population:")
print(f"  missing incoming category: {stats['missing_incoming']}")
print(f"  missing response category: {stats['missing_reply']}")
print(f"  both categories missing: {stats['missing_both']}")
print(f"points with an empty shot list: {stats['empty_points'] / point_n:.1%}")
print(
    "points with at least one eligible transition: "
    f"{stats['points_with_eligible'] / point_n:.1%}"
)

# This assertion guards the link between the reported eligible population and the
# row-level data that will later be divided into training and test matches.
assert len(response_data) == eligible_n
print("\neligible transition table:")
print(response_data.head(10).to_string(index=False))

print("\nopponent-shot counts (row sums):")
print(matrix.sum(axis=1).astype(int))
low = matrix.sum(axis=1)
print("rows with fewer than 30 opponent shots:", list(low[low < 30].index))
print(matrix)

print("\nreproducible audit sample:")
print(pd.DataFrame(examples).to_string(index=False))

# A row of NaNs means that no eligible transition had that incoming bucket; it does
# not represent a uniform response distribution.
row_sums = matrix.sum(axis=1)
prob_matrix = matrix.div(row_sums.replace(0, pd.NA), axis=0)
print("\nconditional response probabilities:")
print(prob_matrix)

Carlos Alcaraz: 222 matches, 36949 points
parser and hitter checks passed
matches: 221
points: 36949
opponent -> Carlos Alcaraz pairs: 50593
points with an empty shot list: 13.5%
points with at least one counted pair: 59.9%
opponent-shot counts (row sums):
opponent_shot
f1     8101
f2     7712
f3     9450
b1     3887
b2     9081
b3    12362
dtype: int64
rows with fewer than 30 opponent shots: []
player_response    f1    f2    f3    b1    b2    b3
opponent_shot                                      
f1               3345  2232  2501     7     6    10
f2               2058  1105  2261   365   719  1204
f3                324   121   386  1491  2330  4798
b1               1902   806  1163     6     4     6
b2               3314  1111  2605   509   707   835
b3               1137   316  1550  2044  2167  5148
                       1st                     2nd  Svr  player_side                                 shots                      hitters                           counted
0          6b38